# LIMINA -- 05. Penilaian Pasar Hari Ini dan Artefak Produk

Menilai emiten yang datanya lengkap (`raw_ingest.symbols_dengan_data_lengkap`)
memakai salah satu dari tiga model, tergantung apa yang berhasil disimpan
notebook 03: Kandidat 1 (regresi logistik) > Kandidat 5 (anomali tanpa
label) > Kandidat 4 (rule-based, tidak perlu model tersimpan).
Menulis `artifacts/scores.json`.

Beda dari notebook 04 (backtest historis): `is_event_90d` di sini selalu 0
(placeholder, belum bisa diketahui) -- jangan dipakai hitung Precision@20.

In [1]:
import sys
from pathlib import Path


def _cari_root(mulai: Path) -> Path:
    for kandidat in [mulai, *mulai.parents]:
        if (kandidat / "limina" / "__init__.py").exists():
            return kandidat
    raise RuntimeError(
        "Tidak menemukan folder 'limina/' di direktori ini atau induknya. "
        "Jalankan notebook dari dalam folder proyek LIMINA."
    )


ROOT = _cari_root(Path.cwd())
sys.path.insert(0, str(ROOT))

import json
from datetime import date

import pandas as pd

from limina import artifacts_io, baselines, config, model_registry, models, raw_ingest

df_qf = pd.read_csv(config.RAW_DIR / f"{config.TABEL_QUARTERLY_FINANCIALS}.csv")
df_dt = pd.read_csv(config.RAW_DIR / f"{config.TABEL_DAILY_TRANSACTION}.csv")
df_dfu = pd.read_csv(config.RAW_DIR / f"{config.TABEL_DAILY_FULL_UNIVERSE_CLOSE}.csv")
df_ff = pd.read_csv(config.RAW_DIR / f"{config.TABEL_FREE_FLOAT_SNAPSHOT}.csv")
df_co = pd.read_csv(config.RAW_DIR / f"{config.TABEL_COMPANY_OVERVIEW}.csv")

df_harga = raw_ingest.gabungkan_harga(df_dt, df_dfu)
peta_sektor = raw_ingest.bangun_peta_sektor(df_ff, df_co)
peta_board = raw_ingest.bangun_peta_board(df_co)
peta_nama = dict(zip(df_co["symbol"], df_co["company_name"])) if "company_name" in df_co.columns else {}

# Cakupan = qf + harga tersedia, bukan union company_overview (notebook 02 bagian 2)
symbols_universe = raw_ingest.symbols_dengan_data_lengkap(df_qf, df_harga)
hari_ini = date.today().strftime("%Y-%m-%d")
print(f"Menilai {len(symbols_universe)} emiten (quarterly_financials + harga tersedia) pada {hari_ini}")

Menilai 19 emiten (quarterly_financials + harga tersedia) pada 2026-09-22


## 1. Bangun potret pasar hari ini

In [2]:
potret_live = raw_ingest.bangun_snapshot_pasar(
    hari_ini, symbols_universe, df_qf, df_harga, peta_sektor, peta_board=peta_board,
)
potret_live["nama"] = potret_live["symbol"].map(peta_nama).fillna("")
potret_live["data_terakhir"] = potret_live["feature_max_source_date"].astype(str)

print(f"{len(potret_live)} baris, {int((potret_live['data_complete'] == 0).sum())} tidak lengkap")

19 baris, 0 tidak lengkap


## 2. Bandingkan dengan riwayat 30 hari lalu (arah_30h, delta_30h)

`riwayat_skor.csv` terkumpul tiap kali notebook ini jalan. Sebelum ada
~30 hari riwayat, seluruh emiten tertulis "stabil" -- bukan bug.

In [3]:
if model_registry.model_tersedia():
    JENIS_MODEL = "kandidat_1"
    model_lr, scaler, median_latih = model_registry.muat_model_terlatih()
    skor_mentah = models.skor_kandidat_1(model_lr, scaler, potret_live, median_latih)
elif model_registry.model_kandidat_5_tersedia():
    JENIS_MODEL = "kandidat_5"
    model_iso, median_latih = model_registry.muat_model_kandidat_5()
    skor_mentah = models.skor_kandidat_5(model_iso, potret_live, median_latih)
else:
    JENIS_MODEL = "rule_based"
    skor_mentah = potret_live.apply(baselines.skor_rule_based_mentah, axis=1)

print(f"Jenis model: {JENIS_MODEL}")
persentil_hari_ini = skor_mentah.rank(pct=True) * 100

if config.PATH_RIWAYAT_SKOR.exists():
    riwayat_skor = pd.read_csv(config.PATH_RIWAYAT_SKOR, parse_dates=["tanggal"])
else:
    riwayat_skor = pd.DataFrame(columns=["symbol", "tanggal", "persentil"])

TOLERANSI_HARI = 5  # +-5 hari di sekitar 30 hari lalu
AMBANG_PERUBAHAN = 5  # poin persentil, di bawah ini "stabil"
target_30h = pd.Timestamp(hari_ini) - pd.Timedelta(days=30)

arah_30h, delta_30h = [], []
for symbol, persentil_now in zip(potret_live["symbol"], persentil_hari_ini):
    riwayat_symbol = riwayat_skor[riwayat_skor["symbol"] == symbol].copy()
    if len(riwayat_symbol) == 0:
        arah_30h.append("stabil")
        delta_30h.append(0.0)
        continue
    riwayat_symbol["jarak"] = (riwayat_symbol["tanggal"] - target_30h).abs()
    terdekat = riwayat_symbol.sort_values("jarak").iloc[0]
    if terdekat["jarak"] > pd.Timedelta(days=TOLERANSI_HARI):
        arah_30h.append("stabil")
        delta_30h.append(0.0)
        continue
    delta = float(persentil_now - terdekat["persentil"])
    delta_30h.append(delta)
    arah_30h.append("naik" if delta > AMBANG_PERUBAHAN else "turun" if delta < -AMBANG_PERUBAHAN else "stabil")

potret_live["arah_30h"] = arah_30h
potret_live["delta_30h"] = delta_30h
print(pd.Series(arah_30h).value_counts())

Jenis model: kandidat_5
stabil    19
Name: count, dtype: int64


## 3. Tulis scores.json

Kandidat 5/rule-based: `indikator_dominan` dari z-score/aturan tetap, bukan
koefisien regresi -- lihat `limina/models.py::kontribusi_kandidat_5` dan
`limina/baselines.py`.

In [4]:
with open(config.PATH_JENDELA) as f:
    jendela = json.load(f)

jumlah_positif_latih = int(pd.read_csv(config.PATH_PANEL)["is_event_90d"].sum()) if config.PATH_PANEL.exists() else 0
cutoff_latih_str = jendela["cutoff_latih"]
output_path = config.ARTIFACTS_DIR / "scores.json"

if JENIS_MODEL == "kandidat_1":
    data_scores = model_registry.hasilkan_scores_json(
        potret_live,
        jumlah_sampel_positif_latih=jumlah_positif_latih,
        dilatih_pada=f"peristiwa sebelum {cutoff_latih_str}",
        k_top=config.K_TOP,
    )
else:
    if JENIS_MODEL == "kandidat_5":
        jenis_model_json = "anomali_kandidat_5"
        df_acuan = potret_live[potret_live["data_complete"] == 1]
        if len(df_acuan) < 2:
            df_acuan = potret_live
        kontribusi_per_emiten = {
            row["symbol"]: models.kontribusi_kandidat_5(df_acuan, row) for _, row in potret_live.iterrows()
        }
        dilatih_pada = "anomali tanpa label, lihat limina/models.py::latih_kandidat_5"
    else:
        jenis_model_json = "rule_based"
        kontribusi_per_emiten = {}
        dilatih_pada = "tidak dilatih -- bobot tetap, lihat limina/baselines.py"

    if config.PATH_KEPUTUSAN.exists():
        with open(config.PATH_KEPUTUSAN) as f:
            keputusan = json.load(f)
        dilatih_pada += f" ({keputusan['keputusan']}: {keputusan['alasan']})"

    df_skor = potret_live.copy()
    df_skor["skor"] = skor_mentah
    df_skor["persentil"] = persentil_hari_ini
    k_efektif = min(config.K_TOP, len(df_skor)) if len(df_skor) else 0
    ambang_persentil = 100 * (1 - k_efektif / len(df_skor)) if len(df_skor) else 0.0
    ambang_nilai = float(skor_mentah.quantile(ambang_persentil / 100)) if len(df_skor) else 0.0
    data_scores = artifacts_io.bangun_scores_json(
        df_skor, kontribusi_per_emiten, jenis_model=jenis_model_json, dilatih_pada=dilatih_pada,
        jumlah_sampel_positif_latih=jumlah_positif_latih,
        ambang_persentil=float(ambang_persentil), ambang_nilai=ambang_nilai,
    )
    artifacts_io.tulis_json(data_scores, output_path)

print(f"scores.json ditulis: {output_path}")
print(f"Cakupan: {data_scores['cakupan']}")
print("\n20 emiten berisiko tertinggi:")
display(
    pd.DataFrame(data_scores["emiten"])
    .sort_values("persentil", ascending=False)
    .head(20)[["symbol", "nama", "sektor", "papan", "status", "persentil", "kategori", "indikator_dominan"]]
)

scores.json ditulis: /home/runner/work/LIMINA-model-train/LIMINA-model-train/artifacts/scores.json
Cakupan: {'total_emiten': 19, 'diberi_skor': 19, 'tidak_dapat_dinilai': 0, 'sudah_ditandai': 0}

20 emiten berisiko tertinggi:


,symbol,nama,sektor,papan,status,persentil,kategori,indikator_dominan
4,BELI.JK,PT Global Digital Niaga Tbk,Technology,New Economy,dinilai,100.000000,Sangat Tinggi,ako_negatif_berturut
3,BBCA.JK,PT Bank Central Asia Tbk.,Financials,Main,dinilai,94.736842,Tinggi,utang_terhadap_aset
7,INET.JK,PT Sinergi Inti Andalan Prima Tbk,Infrastructures,Development,dinilai,89.473684,Tinggi,lapor_terlambat
18,UDNG.JK,PT Agro Bahari Nusantara Tbk,Consumer Non-Cyclicals,Acceleration,dinilai,84.210526,Tinggi,utang_terhadap_aset
12,MLPT.JK,PT Multipolar Technology Tbk,Technology,Development,dinilai,78.947368,Sedang,volatilitas_90d
9,MDIA.JK,PT Intermedia Capital Tbk,Consumer Cyclicals,Watchlist,dinilai,73.684211,Sedang,hari_tanpa_transaksi_90d
8,LUCY.JK,PT Lima Dua Lima Tiga Tbk,Consumer Cyclicals,Acceleration,dinilai,68.421053,Sedang,turun_dari_puncak_90d
13,PACK.JK,PT Abadi Nusantara Hijau Investama Tbk,Basic Materials,Acceleration,dinilai,63.157895,Sedang,utang_terhadap_aset
2,ASII.JK,Astra International Tbk,Industrials,Main,dinilai,57.894737,Sedang,volatilitas_90d
0,AADI.JK,PT Adaro Andalan Indonesia Tbk,Energy,Main,dinilai,52.631579,Sedang,volatilitas_90d


## 4. Tambahkan hari ini ke riwayat_skor.csv (untuk perbandingan 30 hari berikutnya)

In [5]:
baris_baru = pd.DataFrame({
    "symbol": potret_live["symbol"],
    "tanggal": pd.Timestamp(hari_ini),
    "persentil": persentil_hari_ini.values,
})
riwayat_skor_baru = pd.concat([riwayat_skor, baris_baru], ignore_index=True)
riwayat_skor_baru = riwayat_skor_baru.drop_duplicates(subset=["symbol", "tanggal"], keep="last")

config.ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
riwayat_skor_baru.to_csv(config.PATH_RIWAYAT_SKOR, index=False)
print(f"riwayat_skor.csv: {len(riwayat_skor_baru)} baris total setelah menambahkan hari ini")
print("\nSiklus selesai. Jalankan ulang notebook 01-05 secara berkala (harian/mingguan) untuk memperbarui model dan skor.")

riwayat_skor.csv: 38 baris total setelah menambahkan hari ini

Siklus selesai. Jalankan ulang notebook 01-05 secara berkala (harian/mingguan) untuk memperbarui model dan skor.
